# 🌱 Train Beeja-3M on TinyStories (Colab)

Trains the **modern** Beeja-3M (RoPE + RMSNorm + SwiGLU + weight tying) from random
initialization on the TinyStories corpus, then generates samples.

**Before running:** set the runtime to a GPU — *Runtime → Change runtime type → GPU*.

The pipeline checkpoints every 500 steps and is fully resumable, so a Colab
timeout won't lose progress — just re-run the training cell with `--resume`.

Expected result: a 3M-parameter model does **not** write like ChatGPT — it writes
short, mostly-grammatical simple stories. That is the honest ceiling for this size.

## 1. Get the code and install

In [ ]:
!git clone https://github.com/DheerajPranav/beeja-lm.git
%cd beeja-lm
%pip install -q -e .

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Download TinyStories

The validation split (~20 MB) is enough for a first char-level run and downloads in
seconds. For a longer run, add `--full` (the ~2 GB train split) and raise `max_chars`
in the config.

In [ ]:
!python scripts/download_data.py --dataset tinystories

## 3. Train

Trains for 8000 steps (a few hours on a free T4; faster on A100). Loss and a sample
print every 250/500 steps. Checkpoints land in `checkpoints/`.

If the session drops, re-run this cell adding e.g.
`--resume checkpoints/Beeja-3M-TinyStories-step2000.pt` to continue exactly.

In [ ]:
!python -m beeja.train --config configs/beeja-3m-tinystories.yaml --device cuda

## 4. Generate from the trained model

In [ ]:
!python -m beeja.generate \
    --config configs/beeja-3m-tinystories.yaml \
    --checkpoint checkpoints/Beeja-3M-TinyStories-final.pt \
    --prompt 'Once upon a time' --max-new-tokens 300 --temperature 0.8 --device cuda

## 5. Save the checkpoint

Download the final checkpoint (or mount Google Drive and copy it there) so the
trained weights survive the session.

In [ ]:
from google.colab import files
files.download('checkpoints/Beeja-3M-TinyStories-final.pt')